# Code Only

In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
import os
import joblib

data = pd.read_csv(r'E:\Assignments\Evaluation\credit_card_train.csv')

# mapped 'Approved' to 1 and 'Denied' to 0 because using LabelEncoder reverse the values in 'Approved' to 0 and 'Denied' to 1
data['Credit_Card_Issuing'] = data['Credit_Card_Issuing'].map({'Approved': 1, 'Denied': 0})


# Encoding the categorical features
categorical_features = ['Gender', 'Own_Car', 'Own_Housing']

# Will be used to asses the fairness of the model
label_encoders = {}

for col in categorical_features:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    label_encoders[col] = le

# Scaling features for 'Income' and 'Num_Children'
numerical_features = ['Income', 'Num_Children']
scaler = StandardScaler()
data[numerical_features] = scaler.fit_transform(data[numerical_features])


features = ['Num_Children', 'Gender', 'Income', 'Own_Car', 'Own_Housing']
X = data[features]
y = data['Credit_Card_Issuing']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Log Reg model
logreg = LogisticRegression(max_iter=1000, solver='liblinear')

# Using Grid Search for parameters tuning
param_grid = {'C': [0.01, 0.1, 1, 10, 100]}
len_param = len(param_grid['C'])

grid_search = GridSearchCV(
    logreg, param_grid, cv=len_param, scoring='accuracy'
)
grid_search.fit(X_train, y_train)

# Best model
best_logreg = grid_search.best_estimator_

y_pred = best_logreg.predict(X_test)

def predict_credit_approval(input_data):
    df = pd.DataFrame(input_data)
    
    for col in categorical_features:
        le = label_encoders[col]
        df[col] = le.transform(df[col])
    
    df[numerical_features] = scaler.transform(df[numerical_features])
    
    predictions = best_logreg.predict(df[features])
    
    return {"predictions": predictions.tolist()}

In [7]:

# Request Body Examples

request_body = {
    "Num_Children": [1],
    "Gender": ["Male"],
    "Income": [115078],
    "Own_Car": ["No"],
    "Own_Housing": ["Yes"]
} # Approved

# request_body = {
#     "Num_Children": [5],
#     "Gender": ["Female"],
#     "Income": [47506],
#     "Own_Car": ["No"],
#     "Own_Housing": ["No"]
# } # Denied

response = predict_credit_approval(request_body)

print("\nPredictions for the request:")
print(response)


Predictions for the request:
{'predictions': [1]}


# Saving Models

In [ ]:
model_dir = r'E:\Assignments\Evaluation\Models'

# Saving the trained logistic regression model
model_filename = os.path.join(model_dir, f"best_logreg_model.pkl")
joblib.dump(best_logreg, model_filename)

# Saving the label encoders dictionary
label_filename = os.path.join(model_dir, f"label_encoders.pkl")
joblib.dump(label_encoders, label_filename)

# Saving the scaler object
scaler_filename = os.path.join(model_dir, f"scaler.pkl")
joblib.dump(scaler, scaler_filename)